# 07. Registry, Gate, and promotion

Este notebook registra manualmente un artifact `model` ya existente como una Model Version owner-scoped y ejecuta el Quality Gate sobre el mismo run explícito. Los aliases `challenger` y `champion` se asignan manualmente en la UI de MLflow; este notebook no promueve modelos automáticamente.

## Configuración requerida

Configure MLflow como se documenta en el README mediante `MLFLOW_TRACKING_URI` y, si corresponde, las credenciales y el Workspace aprobados. No escriba URLs ni credenciales en este notebook.

Entregue también el contexto académico vigente de InvoiceOps mediante `INVOICEOPS_ORGANIZATION_SLUG`, `INVOICEOPS_OWNER_TYPE`, `INVOICEOPS_OWNER_ID` e `INVOICEOPS_CREATED_BY_RUT`. Seleccione un run existente que tenga el artifact `model` y entregue su identificador mediante `INVOICEOPS_SELECTED_RUN_ID`.

In [ ]:
import os

import mlflow

from invoiceops_ml.gate import run_quality_gate
from invoiceops_ml.mlflow import configure_mlflow, mlflow_config_from_env
from invoiceops_ml.ownership import (
    OwnershipContext,
    owner_registered_model_name,
    set_registered_model_ownership_tags,
)


def required_environment(name: str) -> str:
    value = os.environ.get(name, "").strip()
    if not value:
        raise ValueError(f"{name} must be set from the current InvoiceOps context")
    return value


configure_mlflow(mlflow_config_from_env())
ownership_context = OwnershipContext(
    organization_slug=required_environment("INVOICEOPS_ORGANIZATION_SLUG"),
    owner_type=required_environment("INVOICEOPS_OWNER_TYPE"),
    owner_id=required_environment("INVOICEOPS_OWNER_ID"),
    created_by_rut=required_environment("INVOICEOPS_CREATED_BY_RUT"),
)
selected_run_id = required_environment("INVOICEOPS_SELECTED_RUN_ID")
model_name = owner_registered_model_name(ownership_context)
client = mlflow.MlflowClient()
selected_run = client.get_run(selected_run_id)

if any(
    selected_run.data.tags.get(name) != value
    for name, value in ownership_context.as_tags().items()
):
    raise ValueError("Selected run ownership tags do not match the active ownership context")

matching_versions = [
    version
    for version in client.search_model_versions(f"name='{model_name}'")
    if version.run_id == selected_run_id
]
model_version = (
    matching_versions[0]
    if matching_versions
    else mlflow.register_model(f"runs:/{selected_run_id}/model", model_name)
)
set_registered_model_ownership_tags(model_name, ownership_context, client)

{
    "run_id": selected_run_id,
    "model_name": model_name,
    "model_version": model_version.version,
}

## Designar `challenger` en la UI

Abra el Registered Model indicado arriba en la UI de MLflow, abra la Model Version recién creada y asígnele manualmente el alias `challenger`. Registre en el trabajo del curso el `run_id`, el nombre del Registered Model y el número de Model Version revisados. El alias identifica la versión bajo revisión; no es una aprobación ni una promoción a `champion`.

In [ ]:
run_quality_gate(selected_run_id).as_report()

## Gate es elegibilidad, no promoción

Un resultado `passed: true` sólo confirma que el run satisface los umbrales versionados de elegibilidad. El Gate no registra una Model Version, no asigna aliases, no aprueba una versión y no promueve `challenger` a `champion`. Un resultado FAIL impide continuar con la revisión hasta que exista un nuevo run candidato que cumpla el Gate.

## Promover `champion` manualmente

Sólo después de un PASS del Gate y una decisión explícita de un revisor autorizado, compare en la UI la Model Version `challenger`, sus métricas, artifact, ownership y justificación. Registre la decisión del revisor y luego asigne manualmente el alias `champion` a la versión aprobada en la UI. No asigne `champion` por el PASS del Gate solamente.